# 13 -- Accuracy-vs-Loss Robustness Check

Compares the loss-based Phase-4 predictor results (the report's primary analysis) against
an accuracy-based re-run of the identical analysis, to check the core findings are not an
artifact of the loss-based damage definition (Sec. 4.3). Reproduces the comparison table
and significance-count discussed in report Sec. 5.12.

Source CSVs:
- loss-based: `results/review_response/csv/{normalized_ranks_loss,bootstrap_ci_spearman_loss}.csv`
- accuracy-based: `results/review_response/csv/normalized_ranks.csv`,
  `results/20260816_230437_38678/csv/weight_ablation_canonical_correlation_v2.csv` (CIFAR10),
  `results/20260816_083054_38677/csv/weight_ablation_canonical_correlation_v2.csv` (ImageNet100)


In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."  # report/notebooks -> report -> repo root
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:
RR = f"{REPO}/results/review_response/csv"
ranks_loss = pd.read_csv(f"{RR}/normalized_ranks_loss.csv")
ranks_acc = pd.read_csv(f"{RR}/normalized_ranks.csv")
boot_loss = pd.read_csv(f"{RR}/bootstrap_ci_spearman_loss.csv")

corr_acc_cifar = pd.read_csv(f"{REPO}/results/20260816_230437_38678/csv/weight_ablation_canonical_correlation_v2.csv")
corr_acc_imagenet = pd.read_csv(f"{REPO}/results/20260816_083054_38677/csv/weight_ablation_canonical_correlation_v2.csv")
corr_acc = pd.concat([corr_acc_cifar, corr_acc_imagenet], ignore_index=True)
corr_acc_ptq = corr_acc[corr_acc["stage"] == "PTQ"]


Check 1: is the true top-damage layer identical between the loss-based and accuracy-based definitions, for all 6 PTQ combinations?


In [3]:
top_loss = ranks_loss[ranks_loss["predictor"] == "raw_trh"].set_index(["model", "dataset"])["true_top_layer"]
top_acc = ranks_acc[ranks_acc["predictor"] == "raw_trh"].set_index(["model", "dataset"])["true_top_layer"]
top_compare = pd.DataFrame({"loss_based": top_loss, "accuracy_based": top_acc})
top_compare["match"] = top_compare["loss_based"] == top_compare["accuracy_based"]
n_match = top_compare["match"].sum()
print(f"Top-damage-layer identical in {n_match}/{len(top_compare)} PTQ combinations")
top_compare


Top-damage-layer identical in 6/6 PTQ combinations


loss_based  accuracy_based  match
model               dataset                                           
cnn                 CIFAR10               conv1           conv1   True
                    IMAGENET100           conv1           conv1   True
resnet18_no_weights CIFAR10      layer1.1.conv1  layer1.1.conv1   True
                    IMAGENET100           conv1           conv1   True
resnet50_no_weights CIFAR10                  fc              fc   True
                    IMAGENET100           conv1           conv1   True

Check 2: does any of the 18 score x combination correlation signs flip between loss-based and accuracy-based damage?


In [4]:
loss_rho = boot_loss.set_index(["model", "dataset", "predictor"])["rho"]
acc_rho = corr_acc_ptq.set_index(["model", "dataset", "predictor"])["spearman_rho_abs"]
sign_compare = pd.DataFrame({"loss_rho": loss_rho, "acc_rho": acc_rho}).dropna()
sign_compare["sign_flip"] = np.sign(sign_compare["loss_rho"]) != np.sign(sign_compare["acc_rho"])
n_flips = sign_compare["sign_flip"].sum()
print(f"Sign flips: {n_flips}/{len(sign_compare)}")
sign_compare


Sign flips: 0/18


loss_rho   acc_rho  sign_flip
model               dataset     predictor                               
cnn                 CIFAR10     dwsq        -0.8857 -0.637748      False
                                raw_trh      0.7714  0.927634      False
                                trh_dwsq     0.3714  0.550782      False
                    IMAGENET100 dwsq        -0.8286 -0.942857      False
                                raw_trh      0.5429  0.600000      False
                                trh_dwsq    -0.0857 -0.142857      False
resnet18_no_weights CIFAR10     dwsq        -0.6143 -0.362338      False
                                raw_trh      0.4221  0.292208      False
                                trh_dwsq    -0.2234 -0.159740      False
                    IMAGENET100 dwsq        -0.1909 -0.251962      False
                                raw_trh      0.1026  0.121412      False
                                trh_dwsq    -0.1273 -0.152744      False
resnet50_no_weights CIFAR10     dwsq         0.2496  0.176664      False
                                raw_trh      0.3618  0.205152      False
                                trh_dwsq     0.3536  0.193986      False
                    IMAGENET100 dwsq        -0.1108 -0.006171      False
                                raw_trh      0.1347  0.298767      False
                                trh_dwsq     0.0383  0.178570      False

Check 3: how many of the 18 correlations are significant (p < 0.05) under each damage definition?


In [5]:
loss_p = boot_loss.set_index(["model", "dataset", "predictor"])["p"]
acc_p = corr_acc_ptq.set_index(["model", "dataset", "predictor"])["spearman_p_abs"]
n_sig_loss = (loss_p < 0.05).sum()
n_sig_acc = (acc_p < 0.05).sum()
print(f"Significant (p<0.05) correlations: loss-based {n_sig_loss}/18, accuracy-based {n_sig_acc}/18")

robustness_summary = pd.DataFrame({
    "Prüfung": ["Top-Damage-Layer identisch", "Vorzeichenwechsel", "Signifikant (p<0.05)"],
    "Wert": [f"{n_match}/6", f"{n_flips}/18", f"loss {n_sig_loss}/18, acc {n_sig_acc}/18"],
})
robustness_summary.to_csv(f"{FIG_DIR}/tab_13_robustness_summary.csv", index=False)
robustness_summary


Significant (p<0.05) correlations: loss-based 5/18, accuracy-based 3/18


,Prüfung,Wert
0,Top-Damage-Layer identisch,6/6
1,Vorzeichenwechsel,0/18
2,Signifikant (p<0.05),"loss 5/18, acc 3/18"


Beyond the three numeric checks above, Fig. 5 visualizes the agreement directly: for
each of the 18 score x combination pairs, the loss-based Spearman rho (Sec. 5.6) is
plotted against the accuracy-based rho computed here. Points near the y=x diagonal mean
the two damage definitions agree not just in sign but in magnitude.

-> Fig. 5 in the report, Sec. 5.12. Saved as `figures/fig_05_acc_loss_comparison.pdf`.

In [6]:
fig, ax = plt.subplots(figsize=(4.0, 4.0), layout="constrained")

PRED_LABEL = {"raw_trh": "$S_{raw}$", "dwsq": "$S_{pert}$", "trh_dwsq": "$S_{hawq}$"}
PRED_COLOR = {"raw_trh": "#4C72B0", "dwsq": "#DD8452", "trh_dwsq": "#55A868"}
PRED_MARKER = {"raw_trh": "o", "dwsq": "^", "trh_dwsq": "D"}

plot_df = sign_compare.reset_index()

for predictor, group in plot_df.groupby("predictor"):
    ax.scatter(group["loss_rho"], group["acc_rho"], s=45, marker=PRED_MARKER[predictor],
               color=PRED_COLOR[predictor], edgecolors="white", linewidths=0.4,
               label=PRED_LABEL[predictor], zorder=3)

ax.plot([-1, 1], [-1, 1], color="grey", linestyle="--", linewidth=1.0, zorder=1)
ax.axhline(0.0, color="lightgrey", linewidth=0.6, zorder=0)
ax.axvline(0.0, color="lightgrey", linewidth=0.6, zorder=0)

ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_xlabel(r"Spearman $\rho$, loss-basiert (Sec. 5.6)", fontsize=8.5)
ax.set_ylabel(r"Spearman $\rho$, accuracy-basiert", fontsize=8.5)
ax.tick_params(labelsize=7.5)
ax.legend(fontsize=7.5, loc="upper left", frameon=False)
ax.set_aspect("equal")

fig.savefig(f"{FIG_DIR}/fig_05_acc_loss_comparison.pdf")
fig.savefig(f"{FIG_DIR}/fig_05_acc_loss_comparison.png", dpi=200)
plt.close(fig)

## Output

- `figures/tab_13_robustness_summary.csv` -- summary table underlying report Sec. 5.12
- `figures/fig_05_acc_loss_comparison.pdf` -- Fig. 5 of the report (embedded in `sections/05_results.tex`)
- `figures/fig_05_acc_loss_comparison.png` -- PNG copy for quick preview